# Qwen3.8-27B: TP2 versus PP2 on A100 SXM

**Prepared only — no GPU run has been performed.** This notebook previews the
experiment and analyzes saved artifacts. Running all cells never provisions
hardware or runs a benchmark. See [running instructions](qwen38-27b-parallel.md).

Fixed: BF16, CUDA graphs, chunked prefill, prefix caching, 262,144-token server
context. Vary maximum active sequences (4–128) and client concurrency (8–128).
[Official model card](https://huggingface.co/Qwen/Qwen3.8-27B).

The four workloads use up to 15,360 input tokens; they do not stress full 256K
prompts. Each benchmark starts with an empty prefix cache. Cache ratios include
warmups/probe, while throughput and latency describe the timed requests.
Random prompts measure incidental reuse, not a controlled high-hit workload.

In [1]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import display
from llm_serving.schemas import ExperimentProfile
from llm_serving.matrix import expand_matrix
from llm_serving.workloads import expand_workloads
from llm_serving.analysis import discover_runs, history_rows, performance_rows

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'experiment_profiles').is_dir())
profile = ExperimentProfile.model_validate(yaml.safe_load((ROOT / 'experiment_profiles/qwen38-27b-parallel.yaml').read_text()))
cases = expand_matrix(profile)
workloads = expand_workloads(profile.benchmark)
plan = pd.DataFrame([dict(case_id=c.case_id, tp=c.tensor_parallel, pp=c.pipeline_parallel,
                          max_num_seqs=c.max_num_seqs, cuda_graphs=c.cuda_graphs,
                          max_model_len=profile.server.max_model_len,
                          benchmarks=len(workloads)) for c in cases])
display(plan)
print(f'{len(cases)} server cases; {len(cases) * len(workloads)} benchmark invocations; no hardware allocated by this notebook.')

,case_id,tp,pp,max_num_seqs,cuda_graphs,max_model_len,benchmarks
0,case-00-cg1-cp1-tp2-pp1-dp1-bs4-rep0,2,1,4,True,262144,20
1,case-01-cg1-cp1-tp2-pp1-dp1-bs8-rep0,2,1,8,True,262144,20
2,case-02-cg1-cp1-tp2-pp1-dp1-bs16-rep0,2,1,16,True,262144,20
3,case-03-cg1-cp1-tp2-pp1-dp1-bs32-rep0,2,1,32,True,262144,20
4,case-04-cg1-cp1-tp2-pp1-dp1-bs64-rep0,2,1,64,True,262144,20
5,case-05-cg1-cp1-tp2-pp1-dp1-bs128-rep0,2,1,128,True,262144,20
6,case-06-cg1-cp1-tp1-pp2-dp1-bs4-rep0,1,2,4,True,262144,20
7,case-07-cg1-cp1-tp1-pp2-dp1-bs8-rep0,1,2,8,True,262144,20
8,case-08-cg1-cp1-tp1-pp2-dp1-bs16-rep0,1,2,16,True,262144,20
9,case-09-cg1-cp1-tp1-pp2-dp1-bs32-rep0,1,2,32,True,262144,20


12 server cases; 240 benchmark invocations; no hardware allocated by this notebook.


In [2]:
output_root = Path(os.environ.get('LLM_SERVING_OUTPUT_ROOT', str(ROOT / 'output')))
runs = discover_runs(output_root, profile_name=profile.name)
selected_ids = set(filter(None, (x.strip() for x in os.environ.get('LLM_SERVING_RUN_IDS', '').split(','))))
if selected_ids:
    runs = [r for r in runs if r['run_id'] in selected_ids]
display(pd.DataFrame(history_rows(runs)))
rows = pd.DataFrame(performance_rows(runs))
if rows.empty:
    print('No measured runs selected. All performance and cache results remain unknown.')
else:
    rows['parallelism'] = rows.apply(lambda r: f"TP{r.tensor_parallel}/PP{r.pipeline_parallel}", axis=1)
    rows['workload_family'] = rows.workload_slug.str.replace(r'-c\d+$', '', regex=True)
    display(rows[['run_id', 'case_status', 'parallelism', 'max_num_seqs', 'max_concurrency',
                  'workload_family', 'output_throughput_tok_per_s', 'ttft_p95_ms',
                  'itl_p95_ms', 'cache_hit_rate', 'cache_status']])
    display(rows.groupby('run_id').agg(observed_rows=('case_id', 'size'),
        valid_cache_rows=('cache_status', lambda x: x.eq('ok').sum())))
    print(f'Expected {len(cases) * len(workloads)} rows per complete run. Missing cache values are not zeros.')

""


No measured runs selected. All performance and cache results remain unknown.


## Throughput and cache comparison

Plots keep runs and workload families separate. Each line is a server batch
limit; the horizontal axis is client concurrency. A missing cache value stays
missing. Inspect failed cases and coverage before drawing conclusions.
The first case (TP2, batch 4) is the harness baseline; for TP-versus-PP conclusions
compare matching batch sizes and concurrency, rather than baseline deltas.

In [3]:
if not rows.empty:
    successful = rows[rows.case_status.eq('SUCCESS')]
    for (run_id, workload), group in successful.groupby(['run_id', 'workload_family']):
        for metric, label in [('output_throughput_tok_per_s', 'Output tokens/s'),
                              ('ttft_p95_ms', 'TTFT p95 (ms)'),
                              ('cache_hit_rate', 'Prefix cache hit rate (0–1)')]:
            fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
            for ax, mode in zip(axes, ['TP2/PP1', 'TP1/PP2']):
                subset = group[group.parallelism.eq(mode)]
                for batch, line in subset.groupby('max_num_seqs'):
                    line = line.sort_values('max_concurrency')
                    ax.plot(line.max_concurrency, pd.to_numeric(line[metric], errors='coerce'), marker='o', label=f'batch {batch}')
                ax.set(title=mode, xlabel='Client concurrency', ylabel=label)
                ax.set_xticks(profile.benchmark.concurrencies)
                ax.set_xscale('log', base=2)
                ax.grid(alpha=.2)
                if not subset.empty:
                    ax.legend(fontsize=8)
            fig.suptitle(f'{run_id} · {workload}')
            fig.tight_layout()
            plt.show()